# ДЗ 2. Дообучение seq2seq-модели для суммаризации новостей

---

### Зачем это нужно

Суммаризация — это одна из задач преобразования текста, с которой часто сталкиваются NLP-инженеры. На вход модели поступает длинный текст, а на выходе требуется получить его краткое и информативное изложение. Для решения этой задачи используется архитектура encoder-decoder с механизмом cross-attention. Та же архитектура лежит в основе машинного перевода, перефразирования, исправления текста и многих других задач преобразования текста. Разобравшись с ее работой на примере суммаризации, вы поймете общий принцип, который затем переносится на широкий класс генеративных задач.

Не менее важный навык — корректно оценивать качество генерации. В задачах классификации существует единственный правильный ответ, поэтому качество удобно измерять такими метриками, как Accuracy или F1. В задачах генерации ситуация иная: одну и ту же мысль можно выразить множеством разных способов. Поэтому здесь используются специальные метрики, например ROUGE и BLEU, однако их значения не всегда совпадают с человеческой оценкой качества. В этом задании вы не только научитесь вычислять эти метрики, но и поймете их ограничения, чтобы уметь правильно интерпретировать результаты моделей.

### Что вы сделаете

1. Загрузите датасет и подготовите его для обучения seq2seq-модели (блок 1).
2. Получите бейзлайновые результаты: ROUGE/BLEU для дообученной референсной модели и для базовой модели без дообучения (блок 2).
3. Дообучите модель `ruT5-base` на подвыборке и проанализируете, как изменится качество генерации (блок 3).
4. Сравните результаты всех моделей в единой таблице и на конкретных примерах разберете ситуации, в которых автоматические метрики расходятся с человеческой оценкой (блок 4).
5. Сформулируете выводы по результатам эксперимента и объясните, какую роль играет механизм cross-attention в работе модели (блок 5).

> **Сколько займет:** ориентировочно 4—6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

> **Требуется GPU.** В Google Colab выберите Runtime → Change runtime type → GPU. На CPU обучение модели будет занимать слишком много времени. Если GPU недоступен, вы все равно сможете выполнить блоки 1—2 и 4, однако блок 3 (дообучение модели) рассчитан на использование GPU.

### Оценивание

Максимальный балл за работу — 10. Баллы за каждый блок указаны в его заголовке.

## Настройка окружения

In [ ]:
!pip install transformers datasets sentencepiece rouge_score sacrebleu accelerate

In [ ]:
import re
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, set_seed,
)
from datasets import Dataset, DatasetDict, load_dataset
from rouge_score import rouge_scorer
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
set_seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", None)

print("device:", device)
if device == "cpu":
    print("GPU не найден, рекомендуется включить GPU.")

## 1. Данные и токенизация для seq2seq (1.5 балла)

**Датасет:** [`IlyaGusev/gazeta`](https://huggingface.co/datasets/IlyaGusev/gazeta) содержит новости на русском языке: статья `text` и ее краткое саммари `summary`. Ячейка ниже грузит датасет.

**Пояснение.** В моделях seq2seq энкодер сначала преобразует входную статью в последовательность контекстных представлений. Затем декодер генерирует саммари токен за токеном, а механизм cross-attention на каждом шаге позволяет ему обращаться к представлениям, построенным энкодером, и выбирать, какая информация из исходного текста наиболее важна для генерации следующего токена.

Поэтому при подготовке данных необходимо сформировать две последовательности: токенизированный вход (статью) и токенизированную целевую последовательность (саммари). Целевая последовательность записывается в специальное поле `labels` — именно по ней во время обучения вычисляется loss.

Важно учитывать, что для входной и целевой последовательностей обычно задаются разные максимальные длины (`max_length`): статьи, как правило, значительно длиннее своих кратких саммари, поэтому ограничения на число токенов для них различаются.

In [ ]:
BASE_SEQ2SEQ  = "ai-forever/ruT5-base" # базовая модель для дообучения
REF_SUMMARIZER = "IlyaGusev/rut5_base_sum_gazeta" # уже дообученная (в качестве сильного ориентира)

N_TRAIN, N_VAL, N_TEST = 3000, 300, 300 # подвыборка (уменьшите, если не хватает памяти/времени)

def load_summarization_data():
    """Возвращает DatasetDict со сплитами train/val/test"""
      ds = load_dataset("IlyaGusev/gazeta")
      def take(split, n):
          d = ds[split].shuffle(seed=RANDOM_STATE).select(range(min(n, len(ds[split]))))
          return d.remove_columns([c for c in d.column_names if c not in ("text", "summary")])
      out = DatasetDict(train=take("train", N_TRAIN),
                        validation=take("validation", N_VAL),
                        test=take("test", N_TEST))
      print(f'train={len(out["train"])}, val={len(out["validation"])}, test={len(out["test"])}')
      return out

raw = load_summarization_data()

test_texts = list(raw["test"]["text"])
test_refs = list(raw["test"]["summary"])
print("\nПример пары:")
print("СТАТЬЯ :", test_texts[0][:200])
print("САММАРИ :", test_refs[0][:200])

In [ ]:
# Токенизатор базовой модели и лимиты длины
s2s_tok = AutoTokenizer.from_pretrained(BASE_SEQ2SEQ)
MAX_SRC, MAX_TGT = 512, 160

### ✍️ Задание 1.1. Препроцессинг для seq2seq

Реализуйте функцию `preprocess(batch)`, которая токенизирует батч примеров:

- вход — `batch["text"]` с `max_length=MAX_SRC` и `truncation=True`;
- таргет — `batch["summary"]` с `max_length=MAX_TGT` и `truncation=True`, ее id нужно положить в ключ `"labels"` результата.

Затем примените ее ко всем сплитам через `.map(preprocess, batched=True, remove_columns=raw["train"].column_names)` и сохраните в переменную `tokenized`.

In [ ]:
# ✍️ ВАШ КОД (задание 1.1)
def preprocess(batch):
    raise NotImplementedError

### 💬 Задание 1.2. Вопрос на понимание (впишите ответ ниже)

Ответьте в 2—4 предложениях: почему у source (`MAX_SRC=512`) и у target (`MAX_TGT=160`) разные максимальные длины? Что произойдет, если поставить `MAX_SRC` слишком маленьким - например, 64?

> *Ваш ответ:* ...

## 2. Бейзлайн (2 балла)

Прежде чем приступать к дообучению модели, необходимо зафиксировать бейзлайн на тестовой выборке. Это позволит оценить, насколько обучение действительно улучшило качество генерации.

Будут сравниваться две модели:

- **base**: `ai-forever/ruT5-base` **без дообучения**. Эта модель прошла только этап предобучения (pretraining) с использованием задачи восстановления замаскированных фрагментов текста (span corruption). Поэтому она не обучена выполнять суммаризацию и, как правило, будет генерировать саммари низкого качества. Именно этот результат и служит исходной точкой сравнения.
- **reference**: `IlyaGusev/rut5_base_sum_gazeta`, уже **дообученная** на задаче суммаризации новостей. Она используется в качестве эталонной модели, с результатами которой можно сравнить собственное решение. На небольшой обучающей выборке достичь такого же качества, скорее всего, не удастся, однако она позволяет понять, какого уровня можно ожидать от модели после полноценного обучения.

**Пояснение к метрикам.** ROUGE измеряет, насколько полно содержание сгенерированного саммари покрывает эталонный текст. Обычно анализируют ROUGE-1 (совпадение отдельных слов), ROUGE-2 (совпадение биграмм) и ROUGE-L (совпадение самой длинной общей подпоследовательности). BLEU, первоначально предложенная для машинного перевода, оценивает точность совпадения n-грамм предсказания с эталоном и сильнее штрафует за появление лишних слов и фраз.

> **Про ROUGE для русского языка.** Стандартный токенизатор библиотеки `rouge_score` рассчитан преимущественно на английский язык и некорректно обрабатывает кириллицу. В результате значения ROUGE для текстов на русском языке могут оказаться равными нулю. Поэтому в готовом коде используется `RougeScorer` с кастомным русским токенизатором. Не заменяйте его на `evaluate.load("rouge")` без соответствующей настройки токенизации, иначе рассчитанные значения метрики будут некорректными.

In [ ]:
# Подсчет метрик с корректным русским токенизатором. НЕ МЕНЯЙТЕ токенизатор
class RuTokenizer:
    """Токенизатор для ROUGE"""
    def tokenize(self, text):
        return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=RuTokenizer())

def compute_metrics_text(preds, refs):
    """Принимает списки строк, возвращает dict с ROUGE-1/2/L и BLEU."""
    agg = {"rouge1": [], "rouge2": [], "rougeL": []}
    for p, r in zip(preds, refs):
        sc = _scorer.score(r, p)
        for k in agg:
            agg[k].append(sc[k].fmeasure * 100)
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    return {"ROUGE-1": np.mean(agg["rouge1"]), "ROUGE-2": np.mean(agg["rouge2"]),
            "ROUGE-L": np.mean(agg["rougeL"]), "BLEU": bleu}

RESULTS = {}   # сюда складываем метрики каждой модели для итоговой таблицы

### ✍️ Задание 2.1. Генерация саммари и метрики бейзлайна

Реализуйте функцию `generate_summaries(model, tokenizer, texts, batch_size=8, num_beams=4)`: батчами токенизируйте `texts` (с `truncation`, `max_length=MAX_SRC`, `padding=True`, перенос на `device`), вызовите `model.generate(...)` с `num_beams`, `max_new_tokens=MAX_TGT` и `no_repeat_ngram_size=3`, декодируйте и верните список строк-саммари.

Затем посчитайте метрики для двух моделей на `test_texts` и сохраните в `RESULTS`:
- `RESULTS["reference"]` — для `REF_SUMMARIZER`;
- `RESULTS["base"]` — для `BASE_SEQ2SEQ` без дообучения.

In [ ]:
# ✍️ ВАШ КОД (задание 2.1)
@torch.no_grad()
def generate_summaries(model, tokenizer, texts, batch_size=8, num_beams=4):
    raise NotImplementedError

### 💬 Задание 2.2. Интерпретация результатов (впишите ответ ниже)

Используя полученные значения метрик, сравните качество базовой и референсной моделей. Насколько велик разрыв между ними? Объясните, почему базовая модель показывает такие результаты и зачем в машинном обучении вообще фиксируют бейзлайн, даже если заранее ожидают, что он будет значительно слабее.

> *Ваш ответ:* ...

## 3. Дообучение seq2seq (2 балла)

На этом этапе вы дообучите модель `ruT5-base` на подвыборке обучающих данных. Цель задания — не получить наилучшее качество суммаризации, а проследить, как Fine-Tuning влияет на результаты модели. Поскольку обучение проводится всего на нескольких тысячах примеров и занимает одну эпоху, достигнуть качества референсной модели, обученной на полном датасете, не получится - и именно такой результат является ожидаемым.

> Если упираетесь в память или время: уменьшите `N_TRAIN` (в блоке 1), `MAX_SRC`, уменьшите `per_device_train_batch_size` или ограничьте обучение фиксированным числом шагов (`max_steps`) вместо полной эпохи.

> Про `fp16`. Для моделей семейства T5 обучение в `fp16` нередко становится нестабильным из-за переполнения значений активаций, что приводит к появлению `NaN` в loss. Ниже в конфигурации автоматически используется `bf16` (если его поддерживает оборудование), иначе обучение выполняется в `fp32`. Если во время обучения loss становится равным `NaN`, в первую очередь убедитесь, что `fp16` отключен.

In [ ]:
# Модель, collator и функция метрик для Trainer. НЕ МЕНЯЙТЕ compute_metrics
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_SEQ2SEQ).to(device)
collator = DataCollatorForSeq2Seq(s2s_tok, model=model)

def compute_metrics(eval_pred):
    """Для Trainer: декодирует предсказания/таргет и считает ROUGE/BLEU."""
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, s2s_tok.pad_token_id)
    labels = np.where(labels != -100, labels, s2s_tok.pad_token_id)
    dec_preds = s2s_tok.batch_decode(preds, skip_special_tokens=True)
    dec_labels = s2s_tok.batch_decode(labels, skip_special_tokens=True)
    return {k: round(v, 2) for k, v in compute_metrics_text(dec_preds, dec_labels).items()}

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("bf16:", use_bf16)

### ✍️ Задание 3.1. Дообучите модель

1. Допишите недостающие гиперпараметры в `Seq2SeqTrainingArguments` (см. комментарии в ячейке).
2. Создайте объект `Seq2SeqTrainer` и запустите обучение, вызвав `trainer.train()`.

Часть параметров уже задана (в том числе `predict_with_generate=True` и режим обучения `bf16`, если он поддерживается). Вам необходимо выбрать основные гиперпараметры обучения: размер батча, скорость обучения и продолжительность обучения.

In [ ]:
# ✍️ ВАШ КОД (задание 3.1)
args = Seq2SeqTrainingArguments(
    output_dir="out_sum",
    predict_with_generate=True, # без этого eval не сгенерирует саммари и метрики будут пустыми
    bf16=use_bf16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    # ...
)

## 4. Сравнение и анализ ошибок (3 балла)

Это центральный блок домашнего задания. Здесь вы не просто получите значения ROUGE и BLEU, а научитесь **интерпретировать эти метрики**, понимая, почему высокая оценка не всегда означает хорошее саммари, а низкая — не всегда плохое. Именно так обычно оценивают генеративные модели в реальных проектах.

### ✍️ Задание 4.1. Оценка дообученной модели и сравнение результатов

1. Сгенерируйте саммари **дообученной** моделью (`model` после `trainer.train()`) на `test_texts` и посчитайте метрики ROUGE и BLEU.
2. Сохраните полученные метрики в `RESULTS["fine-tuned"]`.
3. Выведите сводную таблицу по всем трем моделям: `pd.DataFrame(RESULTS).T.round(2)`.

In [ ]:
# ✍️ ВАШ КОД (задание 4.1)

### ✍️ Задание 4.2. Анализ качества генерации

Выберите 2—3 примера из тестового набора и для каждого выведите:
- начало исходной статьи;
- эталонное саммари;
- саммари, сгенерированное дообученной моделью;
- значения ROUGE и BLEU для этого примера.

Сравните значения метрик со своей субъективной оценкой качества суммаризации. Есть ли примеры, где высокие метрики соответствуют неудачному саммари или, наоборот, низкие метрики получает вполне удачное саммари? Если да, кратко объясните, почему это произошло.

In [ ]:
# ✍️ ВАШ КОД (задание 4.2)

### 💬 Задание 4.3. Интерпретация результатов

Проанализируйте полученные результаты.

1. Насколько дообученная модель улучшила качество по сравнению с базовой и насколько приблизилась к референсной? Подтвердите ответ значениями основных метрик.
2. Рассмотрите один из примеров из задания 4.2. Совпадает ли оценка качества по ROUGE с вашей субъективной оценкой саммари? Если нет - объясните, почему возникло расхождение. Если да — поясните, что именно хорошо отражает метрика в этом случае.

> *Ваш ответ:*
>
> 1. ...
> 2. ...

## Блок 5. Выводы (1.5 балла)

### 💬 Задание 5.1. Выводы (впишите ответ ниже)

Подведите итоги выполненной работы (5–8 предложений).

В своем ответе отразите следующие вопросы:

- Что показало сравнение базовой, дообученной и референсной моделей? Дало ли дообучение заметный прирост качества и совпадает ли это с вашим впечатлением от сгенерированных саммари?
- Какую роль играет cross-attention в архитектуре seq2seq и почему без него декодеру было бы значительно сложнее формировать качественное саммари?
- Какие ограничения вы заметили у метрик ROUGE и BLEU? Какие способы оценки качества вы дополнительно использовали бы в реальном проекте?
- Какие идеи из этого задания напрямую переносятся на другие задачи условной генерации (машинный перевод, перефразирование)?

> *Ваш ответ:* ...

## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

Это небольшой блок из двух коротких заданий на выбор — можно сделать одно или оба.

### ✍️ Задание Б1. Влияние стратегии декодирования

Сравните, как стратегия декодирования влияет на качество генерации.

1. Сгенерируйте саммари дообученной модели двумя способами:
    - beam search (`num_beams=4`, как в основной части);
    - жадное декодирование (`num_beams=1`, `do_sample=False`).
2. Посчитайте для обоих вариантов ROUGE и BLEU.
3. В 2–3 предложениях объясните, почему beam search обычно позволяет получить более высокие значения метрик.

In [ ]:
# ✍️ ВАШ КОД (задание Б1)

### ✍️ Задание Б2. Что на самом деле измеряет ROUGE?

Проверьте, насколько ROUGE чувствителен к порядку слов.

1. Возьмите 3–5 эталонных саммари и случайным образом перемешайте слова в каждом из них (`random.shuffle`).
2. Посчитайте ROUGE-1 и ROUGE-2 между исходным и перемешанным текстом.
3. В 2–3 предложениях объясните:
    - почему ROUGE-1 обычно уменьшается незначительно;
    - почему ROUGE-2 падает гораздо сильнее;
    - что это говорит о том, какую информацию учитывает каждая из этих метрик.

In [ ]:
# ✍️ ВАШ КОД (задание Б2)